In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import plotly.express as px
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

## Import & Data cleaning

In [ ]:
df_train = pd.read_csv(Path.cwd() / "data" / "raw" / "train.csv", index_col=0)
df_test = pd.read_csv(Path.cwd() / "data" / "raw" / "test.csv", index_col=0)

In [ ]:
# Bands that are completely empty in the training data -> drop and remember them
empty_cols = df_train.columns[df_train.isna().all()].tolist()
df_train = df_train.drop(columns=empty_cols)


def drop_empty_bands(df):
    """Give df exactly df_train's columns: drop extras, add missing bands as 0.

    'Crop' and 'Stage' are never added (targets to be predicted).
    Existing NaN cells are left untouched (filled in step 3).
    """
    ref = [c for c in df_train.columns if c not in ("Crop", "Stage")]
    return df.reindex(columns=ref, fill_value=0)


df_test = drop_empty_bands(df_test)
df_test.shape

In [ ]:


def fill_bands_neighbor_mean(df, step=10, max_radius=50):
    """Fill NaN band values with the mean of nearest valid wavelength neighbors.

    For a missing value at wavelength w, take the valid values at w +/- step;
    if only one exists, use it alone; if both are missing, expand to
    w +/- 2*step, and so on. Falls back to the band mean if no valid
    neighbor exists. Non-band columns are left untouched.
    """
    band_cols = [c for c in df.columns if c.startswith("X") and c[1:].isdigit()]
    pos = {c: j for j, c in enumerate(band_cols)}
    by_wl = {int(c[1:]): c for c in band_cols}
    band_means = df[band_cols].mean().to_dict()

    filled = df.copy()
    values = filled[band_cols].to_numpy(dtype=float).copy()
    rows = np.where(np.isnan(values).any(axis=1))[0]
    for r in rows:
        for j in np.where(np.isnan(values[r]))[0]:
            w = int(band_cols[j][1:])
            replacement = np.nan
            radius = step
            while radius <= max_radius and np.isnan(replacement):
                neighbors = [
                    values[r, pos[by_wl[w_nb]]]
                    for w_nb in (w - radius, w + radius)
                    if w_nb in by_wl
                ]
                neighbors = [v for v in neighbors if not np.isnan(v)]
                if neighbors:
                    replacement = sum(neighbors) / len(neighbors)
                radius += step
            if np.isnan(replacement):
                replacement = band_means[band_cols[j]]
            values[r, j] = replacement
    filled[band_cols] = values
    return filled


df_train = fill_bands_neighbor_mean(df_train)
df_test = fill_bands_neighbor_mean(df_test)
print("remaining NaNs:", int(df_train.isna().sum().sum() + df_test.isna().sum().sum()))

In [ ]:
# Cyclic transformation Monts

def cyclic_transform_month(df):
    """Add cyclic transformation of the month column.
    With growht check hemispher to determinate seasonality! (not implemented)"""
    df['month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    return df.copy()

df_train = cyclic_transform_month(df_train)
df_test = cyclic_transform_month(df_test)


In [ ]:
df_train.head()

## Analysis Features and Target

### Count of Crop and Stage combinations in the training data

In [ ]:
df_counts = df_train.groupby(['Crop','Stage']).agg({'Stage': 'count'})
df_counts.columns = ['count']
df_counts = df_counts.reset_index()
df_counts.columns

In [ ]:
px.bar(df_counts, x='Crop', y='count', color='Stage', barmode='group', title='Count of Crop and Stage combinations in the training data', labels={'count': 'Count', 'Crop': 'Crop', 'Stage': 'Stage'})

In [ ]:
df_train

In [ ]:
band_cols = [col for col in df_train.columns if col.startswith("X") and col[1:].isdigit()]

df_spectra = df_train[band_cols].melt(
    var_name="Wavelength",
    value_name="Value",
)
df_spectra["Wavelength (nm)"] = df_spectra["Wavelength"].str[1:].astype(int)
df_spectra

In [ ]:
band_cols = [col for col in df_train.columns if col.startswith("X") and col[1:].isdigit()]

df_spectra = df_train[["Crop", "Stage", *band_cols]].melt(
    id_vars=["Crop", "Stage"],
    value_vars=band_cols,
    var_name="Wavelength",
    value_name="Value",
)
df_spectra["Wavelength (nm)"] = df_spectra["Wavelength"].str[1:].astype(int)
df_spectra

In [ ]:
from math import ceil

import plotly.graph_objects as go
from plotly.subplots import make_subplots

BANDS = [
    column for column in df_train.columns
    if column.startswith("X") and column[1:].isdigit()
]
WAVELENGTHS = [int(column[1:]) for column in BANDS]
GROUPS = list(df_train.groupby(["Crop", "Stage"], sort=True, observed=True))

N_COLS = 2  # Set to 1, 2, or 3.
if N_COLS not in (1, 2, 3):
    raise ValueError("N_COLS must be 1, 2, or 3")


def make_spectral_figure(title):
    """Create a subplot grid with one panel per observed crop-stage pair."""
    n_cols = min(N_COLS, len(GROUPS))
    n_rows = ceil(len(GROUPS) / n_cols)
    subplot_titles = [
        f"{crop} — {stage} (n={len(group)})"
        for (crop, stage), group in GROUPS
    ]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        shared_xaxes=True,
        shared_yaxes=True,
    )
    fig.update_layout(
        title=title,
        height=max(350, 300 * n_rows),
        width=500 * n_cols,
        showlegend=False,
        hovermode="closest",
    )
    return fig, n_cols

In [ ]:
plot_df.head()

In [ ]:
import plotly.express as px

ordered_bands = sorted(BANDS, key=lambda band: int(band[1:]))

plot_df = (
    df_train[["Crop", "Stage", *ordered_bands]]
    .reset_index(names="id")
    .melt(
        id_vars=["id", "Crop", "Stage"],
        var_name="Band",
        value_name="Reflectance",
    )
)
plot_df["Wavelength"] = plot_df["Band"].str[1:].astype(int)

fig = px.line(
    plot_df[plot_df["Crop"] == "cotton"],
    x="Wavelength",
    y="Reflectance",
    color="Stage",
    #symbol="Stage",
    line_group="id",
    #markers=True,
    hover_name="id",
    hover_data={"Wavelength": True, "Reflectance": True},
    title="Individual spectra by crop and stage",
    height=900,
)

fig.update_xaxes(type="linear", title_text="Wavelength [nm]")
fig.update_yaxes(title_text="Reflectance [%]")
fig.update_traces(opacity=0.2)
fig.show()

### Line plots spectra

In [ ]:
fig, n_cols = make_spectral_figure("Individual spectra by crop and stage")

for group_number, (_, group) in enumerate(GROUPS):
    x_values = []
    y_values = []
    sample_ids = []

    for sample_id, sample in group.iterrows():
        x_values.extend(WAVELENGTHS)
        x_values.append(None)
        y_values.extend(sample[BANDS].tolist())
        y_values.append(None)
        sample_ids.extend([str(sample_id)] * len(BANDS))
        sample_ids.append(None)

    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            customdata=sample_ids,
            mode="lines",
            line={"width": 0.8},
            opacity=0.55,
            hovertemplate=(
                "id=%{customdata}<br>Wavelength=%{x} nm<br>"
                "Reflectance=%{y}<extra></extra>"
            ),
        ),
        row=group_number // n_cols + 1,
        col=group_number % n_cols + 1,
    )

fig.update_xaxes(title_text="Wavelength [nm]")
fig.update_yaxes(title_text="Reflectance [%]")
fig.show()

### Boxplot frequency of reflectance values by crop and stage

In [ ]:
fig, n_cols = make_spectral_figure(
    "Reflectance distribution by wavelength, crop, and stage"
)

ordered_bands = sorted(BANDS, key=lambda band: int(band[1:]))
wavelengths_nm = [int(band[1:]) for band in ordered_bands]
box_width = 0.8 * min(
    right - left for left, right in zip(wavelengths_nm, wavelengths_nm[1:])
)

for group_number, (_, group) in enumerate(GROUPS):
    fig.add_trace(
        go.Box(
            x=wavelengths_nm * len(group),
            y=group[ordered_bands].to_numpy().ravel(),
            width=box_width,
            boxpoints=False,
            hovertemplate="Wavelength=%{x} nm<br>Reflectance=%{y}<extra></extra>",
        ),
        row=group_number // n_cols + 1,
        col=group_number % n_cols + 1,
    )

fig.update_xaxes(
    type="linear",
    title_text="Wavelength [nm]",
    tickmode="array",
    tickvals=wavelengths_nm[::10],
)
fig.update_yaxes(title_text="Reflectance [%]")
fig.show()

### Scatter plot of the mean reflectance for each crop and stage combination

In [ ]:
fig, n_cols = make_spectral_figure("Spectral scatter by crop and stage")

for group_number, (_, group) in enumerate(GROUPS):
    sample_ids = [
        str(sample_id)
        for sample_id in group.index
        for _ in BANDS
    ]

    fig.add_trace(
        go.Scatter(
            x=WAVELENGTHS * len(group),
            y=group[BANDS].to_numpy().ravel(),
            customdata=sample_ids,
            mode="markers",
            marker={"size": 4, "opacity": 0.55},
            hovertemplate=(
                "id=%{customdata}<br>Wavelength=%{x} nm<br>"
                "Reflectance=%{y}<extra></extra>"
            ),
        ),
        row=group_number // n_cols + 1,
        col=group_number % n_cols + 1,
    )

fig.update_xaxes(title_text="Wavelength [nm]")
fig.update_yaxes(title_text="Reflectance [%]")
fig.show()

In [ ]:
#Line Diagramm frequency of the crops and stages

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

band_cols = sorted(
    [c for c in df_train.columns if c.startswith("X") and c[1:].isdigit()],
    key=lambda c: int(c[1:]),
)
wl = np.array([int(c[1:]) for c in band_cols], dtype=float)

# long form WITH sample id (keep this instead of your old df_spectra)
df_spectra = df_train[["id", "Crop", "Stage", *band_cols]].melt(
    id_vars=["id", "Crop", "Stage"],
    value_vars=band_cols,
    var_name="Wavelength",
    value_name="Value",
)
df_spectra["Wavelength (nm)"] = df_spectra["Wavelength"].str[1:].astype(int)
df_spectra["Crop / Stage"] = df_spectra["Crop"] + " / " + df_spectra["Stage"]

groups = sorted(df_spectra["Crop / Stage"].unique())
ncols, nrows = 3, (len(groups) + 2) // 3
fig = make_subplots(
    rows=nrows, cols=ncols, subplot_titles=groups,
    horizontal_spacing=0.04, vertical_spacing=0.08,
)
for i, grp in enumerate(groups):
    sub = (
        df_spectra[df_spectra["Crop / Stage"] == grp]
        .pivot(index="id", columns="Wavelength (nm)", values="Value")
        .sort_index(axis=1)  # numeric nm order -> true scale, gaps kept
    )
    xb = np.empty((len(sub), len(sub.columns) + 1))
    xb[:, :-1] = sub.columns.to_numpy(dtype=float)
    xb[:, -1] = np.nan
    yb = np.empty_like(xb)
    yb[:, :-1] = sub.to_numpy(dtype=float)
    yb[:, -1] = np.nan
    fig.add_trace(
        go.Scattergl(x=xb.ravel(), y=yb.ravel(), mode="lines",
                     line=dict(width=1), opacity=0.35,
                     showlegend=False, hoverinfo="skip"),
        row=i // ncols + 1, col=i % ncols + 1,
    )

fig.update_layout(
    height=300 * nrows,
    title="One line per df_train row (x = true wavelength scale)",
)
fig.update_xaxes(title_text="Wavelength [nm]")
fig.update_yaxes(title_text="Value")
fig.show()

## Loging the data to mlflow

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("AWP2")

In [ ]:
df_train.head()

In [ ]:
df_train = df_train.drop(['Month', 'id'], axis=1)
X = df_train.drop(columns=['Crop', 'Stage'])
y = df_train['Crop']

ct = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), ['AEZ'])
    ],
    remainder='passthrough'
)

# Transform the data
X = ct.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "random_state": 8888,
    "max_depth": 5
}

In [ ]:

mlflow.sklearn.autolog()
clf = DecisionTreeClassifier(**params)
clf.fit(X_train, y_train)

In [ ]:
df_train.head()

In [ ]:
df_train.info(verbose=True, show_counts=True)

In [ ]:
# Print connection information
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: {mlflow.get_experiment_by_name('AWP2')}")

# Test logging
with mlflow.start_run():
    mlflow.log_param("test_param", "test_value")
    print("✓ Successfully connected to MLflow!")

In [ ]:
bands = sorted(
    (column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()),
    key=lambda column: int(column[1:]),
)
wavelengths = pd.Series(
    [int(band[1:]) for band in bands],
    index=bands,
    dtype=float,
)

correlations = df_train[bands].apply(
    lambda spectrum: spectrum.corr(wavelengths),
    axis=1,
)

correlation_df = df_train[["Crop", "Stage"]].copy()
correlation_df["pearson_correlation"] = correlations

display(correlation_df)

In [ ]:
import plotly.express as px

bands = sorted(
    (column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()),
    key=lambda column: int(column[1:]),
)

# Rows are samples; columns are frequencies.
similarity_matrix = df_train[bands].T.corr(method="pearson")
similarity_matrix.index.name = "Sample ID"
similarity_matrix.columns.name = "Sample ID"

display(similarity_matrix)

fig = px.imshow(
    similarity_matrix,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    labels={
        "x": "Sample ID",
        "y": "Sample ID",
        "color": "Pearson correlation",
    },
    title="Pairwise spectral similarity",
)
fig.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

bands = sorted(
    (column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()),
    key=lambda column: int(column[1:]),
)

spectra = df_train[bands]
if spectra.isna().any().any():
    raise ValueError("Handle missing frequency values before calculating cosine similarity.")

similarity_matrix = pd.DataFrame(
    cosine_similarity(spectra),
    index=spectra.index,
    columns=spectra.index,
)
similarity_matrix.index.name = "Sample ID"
similarity_matrix.columns.name = "Sample ID"

similarity_matrix = similarity_matrix.mask(
    np.eye(len(similarity_matrix), dtype=bool)
)
display(similarity_matrix)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

bands = sorted(
    (column for column in df_train.columns
     if column.startswith("X") and column[1:].isdigit()),
    key=lambda column: int(column[1:]),
)

spectra = df_train[bands]
if spectra.isna().any().any():
    raise ValueError("Handle missing frequency values before calculating cosine similarity.")

# Use the existing ID column if present; otherwise use the DataFrame index.
sample_ids = (
    df_train["id"].to_numpy()
    if "id" in df_train.columns
    else df_train.index.to_numpy()
)

# Recompute the matrix with IDs as labels, keeping its order aligned with df_train.
similarity_values = cosine_similarity(spectra)
similarity_matrix = pd.DataFrame(
    similarity_values,
    index=pd.Index(sample_ids, name="ID"),
    columns=pd.Index(sample_ids, name="ID"),
)

row_indices, col_indices = np.triu_indices(len(sample_ids), k=1)
scores = similarity_values[row_indices, col_indices]
valid = ~np.isnan(scores)

crops = df_train["Crop"].to_numpy()
stages = df_train["Stage"].to_numpy()

pairwise_similarities = pd.DataFrame({
    "id x": sample_ids[row_indices[valid]],
    "id y": sample_ids[col_indices[valid]],
    "cosine similarity": scores[valid],
    "crop x": crops[row_indices[valid]],
    "crop y": crops[col_indices[valid]],
    "stage x": stages[row_indices[valid]],
    "stage y": stages[col_indices[valid]],
})

display(pairwise_similarities)

In [ ]:
pairwise_similarities

In [ ]:
import plotly.graph_objects as go

bands = sorted(
    (
        column
        for column in set(df_train.columns) & set(df_test.columns)
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

if not bands:
    raise ValueError("No shared spectral bands found between train and test.")

wavelengths_nm = [int(band[1:]) for band in bands]
train_mean = df_train[bands].mean()
test_mean = df_test[bands].mean()

fig = go.Figure()

for label, mean_spectrum, color in [
    ("Train mean", train_mean, "blue"),
    ("Test mean", test_mean, "orange"),
]:
    x_values = []
    y_values = []

    for index, (wavelength, band) in enumerate(zip(wavelengths_nm, bands)):
        if index > 0 and wavelength - wavelengths_nm[index - 1] > 50:
            x_values.append(None)
            y_values.append(None)

        x_values.append(wavelength)
        y_values.append(mean_spectrum[band])

    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="lines",
            name=label,
            line={"color": color},
            hovertemplate="Wavelength=%{x} nm<br>Mean reflectance=%{y}<extra></extra>",
        )
    )

fig.update_layout(
    title="Mean spectrum: train vs. test",
    xaxis_title="Wavelength [nm]",
    yaxis_title="Mean reflectance [%]",
)
fig.show()